In [1]:
# Berechne die Total Variation Distance für die vier Methoden anhand der Datensätze
import numpy as np
import pandas as pd

from efficient_probit_regression.total_variation_distance import total_variation_distance
from efficient_probit_regression.sampling import compute_leverage_scores, calculate_lewis_weights_exact, calculate_l2_lp_leverage_score, compute_random_evaluations_probabilities_v2, to_density

print("Total Variation Distance")

# load all datasets

%store -r Webspam
%store -r Iris
%store -r KDDCup
%store -r Covertype
%store -r Example2D

p = [1.0, 1.3, 1.5, 1.7, 2.0]

def tvd_table_for_dataset(
    X: np.ndarray,
    dataset_name: str,
    p_values,
    *,
    T_lewis: int = 20,
    m_random: int = 1000,
    rng=2026,
) -> pd.DataFrame:
    """
    Erzeugt einen DataFrame mit allen 6 paarweisen TVDs der 4 Methoden
    (lev, lewis, l2_lp, random) für einen Datensatz und alle p-Werte.
    """

    rows = []

    for p in p_values:
        lev = compute_leverage_scores(X, p=p)
        lw  = calculate_lewis_weights_exact(X, p=p, T=T_lewis)
        l2lp = calculate_l2_lp_leverage_score(X, p=p)[1]
        rnd = compute_random_evaluations_probabilities_v2(X, m=m_random, p=p, rng=rng)

        # Dichten vorbereiten
        lev_d = to_density(lev)
        lw_d  = to_density(lw)

        # 6 Paar-Vergleiche
        tvd_lev_lewis   = round(total_variation_distance(lev, lw, normalize=True), 4)
        tvd_lev_l2_lp   = round(total_variation_distance(lev_d, l2lp), 4)
        tvd_lev_random  = round(total_variation_distance(lev_d, rnd), 4)
        tvd_l2_lp_lewis = round(total_variation_distance(l2lp, lw_d), 4)
        tvd_lewis_rand  = round(total_variation_distance(lw_d, rnd), 4)
        tvd_l2_lp_rand  = round(total_variation_distance(l2lp, rnd), 4)

        rows.append({
            "dataset": dataset_name,
            "p": float(p),
            "lev - lewis": float(tvd_lev_lewis),
            "lev - l2lp": float(tvd_lev_l2_lp),
            "lev - random": float(tvd_lev_random),
            "l2lp - lewis": float(tvd_l2_lp_lewis),
            "lewis - random": float(tvd_lewis_rand),
            "l2lp - random": float(tvd_l2_lp_rand),
        })

    df = pd.DataFrame(rows).sort_values("p").reset_index(drop=True)
    return df

datasets = {
    "Iris": Iris.X,
    "Example2D": Example2D.X,
    "Webspam": Webspam.X,
    "KDDCup": KDDCup.X,
    "Covertype": Covertype.X,
}

tvd_dfs = {}

for name, X in datasets.items():
    df = tvd_table_for_dataset(
        X,
        dataset_name=name,
        p_values=p,
        T_lewis=20,
        m_random=1000,
        rng=2026
    )
    tvd_dfs[name] = df

    print(f"\n=== {name} ===")
    display(df)

Total Variation Distance

=== Iris ===


,dataset,p,lev - lewis,lev - l2lp,lev - random,l2lp - lewis,lewis - random,l2lp - random
0,Iris,1.0,0.0355,0.0105,0.0963,0.0312,0.0987,0.0996
1,Iris,1.3,0.0289,0.0143,0.1164,0.0233,0.1205,0.1218
2,Iris,1.5,0.0223,0.0152,0.1297,0.0171,0.1334,0.1357
3,Iris,1.7,0.0144,0.0129,0.1426,0.0106,0.1452,0.1478
4,Iris,2.0,0.0000,0.0000,0.1614,0.0000,0.1614,0.1614



=== Example2D ===


,dataset,p,lev - lewis,lev - l2lp,lev - random,l2lp - lewis,lewis - random,l2lp - random
0,Example2D,1.0,0.0420,0.0123,0.1061,0.0342,0.0899,0.1033
1,Example2D,1.3,0.0354,0.0171,0.1335,0.0249,0.1173,0.1291
2,Example2D,1.5,0.0281,0.0184,0.1501,0.0174,0.1361,0.1450
3,Example2D,1.7,0.0185,0.0158,0.1648,0.0104,0.1548,0.1601
4,Example2D,2.0,0.0000,0.0000,0.1825,0.0000,0.1825,0.1825



=== Webspam ===


,dataset,p,lev - lewis,lev - l2lp,lev - random,l2lp - lewis,lewis - random,l2lp - random
0,Webspam,1.0,0.2692,0.0015,0.1117,0.2679,0.2664,0.1116
1,Webspam,1.3,0.2625,0.0080,0.1469,0.2552,0.2713,0.1468
2,Webspam,1.5,0.2394,0.0206,0.1754,0.2200,0.2734,0.1764
3,Webspam,1.7,0.1854,0.0394,0.2132,0.1475,0.2745,0.2183
4,Webspam,2.0,0.0000,0.0000,0.3017,0.0000,0.3017,0.3017



=== KDDCup ===


,dataset,p,lev - lewis,lev - l2lp,lev - random,l2lp - lewis,lewis - random,l2lp - random
0,KDDCup,1.0,0.4336,0.0031,0.2399,0.4306,0.5475,0.2399
1,KDDCup,1.3,0.3690,0.0145,0.2738,0.3549,0.5003,0.2732
2,KDDCup,1.5,0.2964,0.0317,0.2910,0.2655,0.4479,0.2897
3,KDDCup,1.7,0.1948,0.0474,0.3003,0.1484,0.3736,0.2999
4,KDDCup,2.0,0.0000,0.0000,0.2945,0.0000,0.2945,0.2945



=== Covertype ===


,dataset,p,lev - lewis,lev - l2lp,lev - random,l2lp - lewis,lewis - random,l2lp - random
0,Covertype,1.0,0.3194,0.0011,0.1035,0.3183,0.3652,0.1042
1,Covertype,1.3,0.2695,0.0061,0.1534,0.2634,0.3732,0.1574
2,Covertype,1.5,0.2159,0.0151,0.2040,0.2009,0.3790,0.2149
3,Covertype,1.7,0.1451,0.0268,0.2725,0.1183,0.3849,0.2925
4,Covertype,2.0,0.0346,0.0170,0.4078,0.0176,0.3942,0.4011


In [5]:
import pandas as pd
from pathlib import Path

df_all = pd.concat(tvd_dfs.values(), ignore_index=True)

# fürs Anzeigen optional runden: alle Spalten außer dataset und p
df_all_show = df_all.copy()
value_cols = [c for c in df_all_show.columns if c not in ("dataset", "p")]

for c in value_cols:
    df_all_show[c] = df_all_show[c].astype(float).round(8)

print("\n=== ALL DATASETS (combined) ===")
display(df_all_show)

out_dir = Path("Tabellen")
out_dir.mkdir(exist_ok=True, parents=True)
out_csv = "tvd_all_methods.csv"
df_all.to_csv(out_dir / out_csv, index=False, float_format="%.10f")
print("Saved:", out_csv)


=== ALL DATASETS (combined) ===


,dataset,p,lev - lewis,lev - l2lp,lev - random,l2lp - lewis,lewis - random,l2lp - random
0,Iris,1.0,0.0355,0.0105,0.0963,0.0312,0.0987,0.0996
1,Iris,1.3,0.0289,0.0143,0.1164,0.0233,0.1205,0.1218
2,Iris,1.5,0.0223,0.0152,0.1297,0.0171,0.1334,0.1357
3,Iris,1.7,0.0144,0.0129,0.1426,0.0106,0.1452,0.1478
4,Iris,2.0,0.0000,0.0000,0.1614,0.0000,0.1614,0.1614
5,Example2D,1.0,0.0420,0.0123,0.1061,0.0342,0.0899,0.1033
6,Example2D,1.3,0.0354,0.0171,0.1335,0.0249,0.1173,0.1291
7,Example2D,1.5,0.0281,0.0184,0.1501,0.0174,0.1361,0.1450
8,Example2D,1.7,0.0185,0.0158,0.1648,0.0104,0.1548,0.1601
9,Example2D,2.0,0.0000,0.0000,0.1825,0.0000,0.1825,0.1825


Saved: tvd_all_methods.csv
